In [1]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
import os  # NEW

# ---------- symbolic definitions ----------
x_sym = sp.symbols('x')
f_sym = x_sym**3 - 7*x_sym**2 + 14*x_sym - 5
fprime_sym = sp.diff(f_sym, x_sym)

# numeric functions
f = sp.lambdify(x_sym, f_sym, 'numpy')
fprime = sp.lambdify(x_sym, fprime_sym, 'numpy')
# ----- compute real root(s) and critical points -----
roots_sym = sp.nroots(f_sym)
real_roots = [complex(r) for r in roots_sym if abs(sp.im(r)) < 1e-10]
real_roots = [r.real for r in real_roots]

crit_points_sym = sp.nroots(sp.Eq(fprime_sym, 0))
real_crit_points = [complex(c) for c in crit_points_sym if abs(sp.im(c)) < 1e-10]
real_crit_points = [c.real for c in real_crit_points]

def newton_raphson_single(x0, max_iter=50, precision=1e-8):
    x = float(x0)

    xs = [x]
    fs_vals = [f(x)]
    fps_vals = [fprime(x)]
    iters_used = 0
    converged = False

    # (iteration index, x_k, f(x_k), f'(x_k), relative error)
    details = [(0, x, f(x), fprime(x), None)]

    for k in range(max_iter):
        fx = f(x)
        fpx = fprime(x)
        if fpx == 0:
            iters_used = k + 1
            break

        x_new = x - fx / fpx
        fx_new = f(x_new)
        fpx_new = fprime(x_new)

        rel_err = abs(x_new - x) / max(abs(x_new), 1e-16)

        xs.append(x_new)
        fs_vals.append(fx_new)
        fps_vals.append(fpx_new)
        iters_used = k + 1
        details.append((k + 1, x_new, fx_new, fpx_new, rel_err))

        if rel_err < precision:
            converged = True
            x = x_new
            break

        x = x_new

    return {
        "x0": x0,
        "xs": xs,
        "fs": fs_vals,
        "fps": fps_vals,
        "iters": iters_used,
        "converged": converged,
        "details": details
    }
